# 🔬 PaliGemma Multi-Model Evaluation - UFV Campus Dataset

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gómez Soteres  
**Universidad Francisco de Vitoria**

---

This notebook evaluates **ALL PaliGemma models** on the **UFV Campus dataset** with multiple question types.

## Experiment Setup:
- **Dataset**: UFV Campus only (automatically filtered)
- **Models**: All models from `config/models.json` (13 models)
- **Questions**: Binary (yes/no, up/down, open/closed) and Element identification
- **Expected Answers**: Per-image specific (loaded from `ufv_expected_answers.json`)

## Metrics Evaluated:
- **Exact Match**: For binary/closed questions (yes/no, up/down, open/closed)
- **Keyword Recall**: For element identification questions
- **BERTScore**: For caption questions (semantic similarity)
- **Throughput**: Images per second
- **Inference Time**: Performance metric
- **VRAM Usage**: GPU memory consumption
- **Confusion Matrices**: YES/NO, OPEN/CLOSED, UP/DOWN, Navigation Elements

## Output:
- CSV file with comparison of all models (one row per model)
- Individual results and metrics for each model
- Confusion matrices for each model

## 1️⃣ Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 accelerate bitsandbytes
!pip install -q torch torchvision
!pip install -q pandas numpy tqdm Pillow matplotlib seaborn

!pip install -q bert_score  # For BERTScore metric

print("✅ All packages installed successfully!")

In [ ]:
# Mount Google Drive (for data only)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted (for data access)!")

In [ ]:
# Setup project - Clone from GitHub (Private Repo)
import os
import sys
import subprocess

# ============================================================
# CONFIGURATION - PRIVATE REPO
# ============================================================
GITHUB_USER = "AlbertoGomezS"
GITHUB_REPO = "tfg-AlbertoGomez"
LOCAL_PROJECT_PATH = '/content/tfg'

# GitHub Personal Access Token (create at: https://github.com/settings/tokens)
# Required scopes: repo (full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # <-- REPLACE THIS

# ============================================================

# Clone or update from GitHub
if os.path.exists(LOCAL_PROJECT_PATH):
    # Update existing repo
    os.chdir(LOCAL_PROJECT_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("✅ Repository updated from GitHub")
    else:
        print(f"⚠️ Git pull issue: {result.stderr}")
else:
    # Clone with authentication (private repo)
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    result = subprocess.run(['git', 'clone', clone_url, LOCAL_PROJECT_PATH], capture_output=True, text=True)
    
    if result.returncode == 0:
        os.chdir(LOCAL_PROJECT_PATH)
        print(f"✅ Repository cloned from GitHub (private)")
    else:
        print(f"❌ Clone failed. Check your GITHUB_TOKEN")
        print(f"   Error: {result.stderr}")
        raise Exception("Could not clone repository - check token")

# Add src to path
sys.path.insert(0, LOCAL_PROJECT_PATH)

print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# Import project modules
from src.model_loader import load_paligemma, unload_model, print_gpu_memory, get_model_info
from src.dataset_loader import (
    load_questions_config,
    load_dataset,
    create_evaluation_items,
    print_dataset_summary,
    load_per_image_answers
)
from src.inference import (
    run_inference_on_dataset,
    print_inference_summary,
    results_to_dict
)
from src.metrics import (
    calculate_all_metrics,
    print_metrics_summary,
    get_gpu_memory_usage,
    reset_gpu_memory_stats
)

import json
import pandas as pd
from datetime import datetime

print("✅ All modules imported successfully!")

## 2️⃣ Configuration

In [ ]:
# ============================================================
# CONFIGURATION - PATHS
# ============================================================

# Data is stored in Google Drive (too large for Git)
DATA_PATH = '/content/drive/MyDrive/tfg/data/UFV Campus'  # Only UFV Campus dataset

# Config and src are cloned from GitHub
CONFIG_PATH = '/content/tfg/config'

# Results stored in Colab /content directory (local)
RESULTS_PATH = '/content/results/all_models_experiment'

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verify data path exists
if not os.path.exists(DATA_PATH):
    print(f"⚠️ Data path not found: {DATA_PATH}")
    print("   Please upload your UFV Campus data folder to Google Drive")
else:
    print(f"✅ Data path found: {DATA_PATH}")
    
print(f"📁 Config path: {CONFIG_PATH}")
print(f"📁 Results path: {RESULTS_PATH}")

In [ ]:
# Load configurations
questions_config = load_questions_config(f"{CONFIG_PATH}/questions.json")

# Load per-image expected answers
per_image_answers = load_per_image_answers(f"{CONFIG_PATH}/ufv_expected_answers.json")

# Load models configuration
with open(f"{CONFIG_PATH}/models.json", 'r') as f:
    models_config = json.load(f)

models_to_test = models_config["models"]

print(f"✅ Loaded {len(questions_config['categories'])} question categories")
print(f"✅ Loaded expected answers for {len(per_image_answers)} images")
print(f"✅ Loaded {len(models_to_test)} models to evaluate")

In [ ]:
# Load UFV Campus dataset
samples = load_dataset(DATA_PATH, questions_config, load_images=False)

# Print dataset summary
print_dataset_summary(samples)

# Create evaluation items (image + question pairs)
eval_items = create_evaluation_items(samples, questions_config, per_image_answers=per_image_answers)

print(f"\n📊 Total evaluation items: {len(eval_items)}")
print(f"   (Each image has multiple questions depending on category)")

## 3️⃣ HuggingFace Login

In [ ]:
# Login to HuggingFace (required for PaliGemma access)
from huggingface_hub import login

# IMPORTANT: Replace with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the PaliGemma license

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"  # <-- REPLACE THIS

if HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN":
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️ Please set your HuggingFace token above")
    print("   1. Get your token from: https://huggingface.co/settings/tokens")
    print("   2. Accept the PaliGemma license")
    print("   3. Replace 'YOUR_HUGGINGFACE_TOKEN' with your actual token")

## 4️⃣ Evaluation Function

In [ ]:
def evaluate_single_model(model_info, eval_items, results_path):
    """
    Evaluate a single model and return results.
    
    Args:
        model_info: Dictionary with model information from models.json
        eval_items: List of evaluation items
        results_path: Path to save results
    
    Returns:
        Dictionary with model info, metrics, and results
    """
    model_id = model_info["model_id"]
    model_desc = model_info["description"]
    
    print(f"\n{'='*60}")
    print(f"🔬 Evaluating: {model_desc}")
    print(f"   Model ID: {model_id}")
    print(f"{'='*60}\n")
    
    try:
        # Reset GPU memory stats
        reset_gpu_memory_stats()
        
        # Load model
        print("🔄 Loading model...")
        model, processor = load_paligemma(model_id)
        
        # Run inference
        print("\n🔬 Running inference...")
        results = run_inference_on_dataset(
            model,
            processor,
            eval_items,
            max_new_tokens=100,
            verbose=False  # Set to True for detailed output
        )
        
        # Convert to dict for metrics
        results_dict = results_to_dict(results)
        
        # Calculate metrics
        vram_usage = get_gpu_memory_usage()
        metrics = calculate_all_metrics(results_dict, vram_usage=vram_usage)
        
        # Print summary
        print_metrics_summary(metrics)
        
        # Save individual model results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name_safe = model_id.replace("/", "_").replace("-", "_")
        
        # Save raw results
        results_file = f"{results_path}/{model_name_safe}_results_{timestamp}.json"
        with open(results_file, 'w') as f:
            json.dump(results_dict, f, indent=2)
        
        # Save metrics
        metrics_file = f"{results_path}/{model_name_safe}_metrics_{timestamp}.json"
        with open(metrics_file, 'w') as f:
            json.dump(metrics, f, indent=2)
        
        print(f"\n💾 Results saved to: {results_file}")
        print(f"💾 Metrics saved to: {metrics_file}")
        
        # Unload model to free memory
        print("\n🧹 Unloading model...")
        unload_model(model, processor)
        
        return {
            "model_info": model_info,
            "metrics": metrics,
            "results": results_dict,
            "status": "success"
        }
        
    except Exception as e:
        print(f"\n❌ Error evaluating {model_id}: {e}")
        import traceback
        traceback.print_exc()
        return {
            "model_info": model_info,
            "status": "error",
            "error": str(e)
        }

In [ ]:
# Evaluate all models
print(f"🚀 Starting evaluation of {len(models_to_test)} models")
print(f"📊 Total items to evaluate per model: {len(eval_items)}")
print("=" * 60)

all_results = {}

for i, model_info in enumerate(models_to_test, 1):
    print(f"\n\n{'#'*60}")
    print(f"# Model {i}/{len(models_to_test)}")
    print(f"{'#'*60}")
    
    result = evaluate_single_model(model_info, eval_items, RESULTS_PATH)
    all_results[model_info["model_id"]] = result
    
    # Small delay to let GPU settle
    import time
    time.sleep(2)

print(f"\n\n{'='*60}")
print("✅ ALL MODELS EVALUATED")
print(f"{'='*60}")

## 6️⃣ Generate Comparison CSV

In [ ]:
# Generate CSV with comparison of all models
comparison_data = []

for model_id, result in all_results.items():
    if result.get("status") != "success":
        # Skip failed models
        continue
    
    model_info = result["model_info"]
    metrics = result["metrics"]
    overall = metrics.get("overall", {})
    
    # Extract confusion matrix metrics
    cm = metrics.get("confusion_matrices", {})
    
    # Binary YES/NO metrics
    binary_yes_no = cm.get("binary_yes_no", {}).get("global", {})
    binary_metrics = binary_yes_no.get("metrics", {}) if binary_yes_no else {}
    binary_cm = binary_yes_no.get("confusion_matrix", {}) if binary_yes_no else {}
    
    # State OPEN/CLOSED metrics
    state_open_closed = cm.get("state_open_closed", {}).get("global", {})
    state_oc_metrics = state_open_closed.get("metrics", {}) if state_open_closed else {}
    
    # State UP/DOWN metrics
    state_up_down = cm.get("state_up_down", {}).get("global", {})
    state_ud_metrics = state_up_down.get("metrics", {}) if state_up_down else {}
    
    # Navigation elements metrics
    nav_elements = cm.get("navigation_elements", {}).get("global", {})
    nav_metrics = nav_elements.get("metrics", {}) if nav_elements else {}
    
    row = {
        "Model ID": model_id,
        "Description": model_info.get("description", ""),
        "Version": model_info.get("version", ""),
        "Size": model_info.get("size", ""),
        "Type": model_info.get("type", ""),
        "Resolution": model_info.get("resolution", ""),
        
        # Overall metrics
        "Exact Match": overall.get("exact_match", 0),
        "Keyword Recall": overall.get("keyword_recall", 0),
        "BERTScore Recall": overall.get("bertscore_recall", 0),
        "BERTScore F1": overall.get("bertscore_f1", 0),
        "BERTScore Precision": overall.get("bertscore_precision", 0),
        
        # Performance metrics
        "Throughput (images/sec)": overall.get("throughput", 0),
        "Avg Inference Time (s)": overall.get("avg_inference_time", 0),
        "Total Inference Time (s)": overall.get("total_inference_time", 0),
        "VRAM Usage (GB)": overall.get("vram_max_allocated_gb", 0),
        
        # Output length
        "Avg Output Length (words)": overall.get("avg_output_length", 0),
        "Max Output Length (words)": overall.get("max_output_length", 0),
        
        # Confusion Matrix - Binary YES/NO
        "Binary YES/NO Accuracy": binary_metrics.get("accuracy", 0),
        "Binary YES/NO Precision": binary_metrics.get("precision", 0),
        "Binary YES/NO Recall": binary_metrics.get("recall", 0),
        "Binary YES/NO F1": binary_metrics.get("f1", 0),
        "Binary YES/NO TP": binary_cm.get("TP", 0),
        "Binary YES/NO FP": binary_cm.get("FP", 0),
        "Binary YES/NO TN": binary_cm.get("TN", 0),
        "Binary YES/NO FN": binary_cm.get("FN", 0),
        
        # Confusion Matrix - State OPEN/CLOSED
        "State OPEN/CLOSED Accuracy": state_oc_metrics.get("accuracy", 0),
        
        # Confusion Matrix - State UP/DOWN
        "State UP/DOWN Accuracy": state_ud_metrics.get("accuracy", 0),
        
        # Confusion Matrix - Navigation Elements
        "Navigation Elements Accuracy": nav_metrics.get("accuracy", 0),
    }
    
    comparison_data.append(row)

# Create DataFrame and save
df_comparison = pd.DataFrame(comparison_data)

# Sort by model ID for consistency
df_comparison = df_comparison.sort_values("Model ID")

# Save CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comparison_file = f"{RESULTS_PATH}/all_models_comparison_{timestamp}.csv"
df_comparison.to_csv(comparison_file, index=False, encoding='utf-8')

print(f"💾 Comparison CSV saved to: {comparison_file}")
print(f"   Total models compared: {len(df_comparison)}")
print(f"\n📊 Preview:")
print(df_comparison[["Model ID", "Description", "Exact Match", "Keyword Recall", "Throughput (images/sec)", "VRAM Usage (GB)"]].head(10))

## 7️⃣ Results Summary

In [ ]:
# Display summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

successful_models = [r for r in all_results.values() if r.get("status") == "success"]
failed_models = [r for r in all_results.values() if r.get("status") != "success"]

print(f"\n✅ Successful evaluations: {len(successful_models)}/{len(all_results)}")
if failed_models:
    print(f"❌ Failed evaluations: {len(failed_models)}")
    for failed in failed_models:
        print(f"   - {failed['model_info']['model_id']}: {failed.get('error', 'Unknown error')}")

if successful_models:
    print(f"\n📊 Metrics Summary (across all successful models):")
    print("-" * 60)
    
    # Calculate averages
    exact_matches = [r["metrics"]["overall"].get("exact_match", 0) for r in successful_models]
    keyword_recalls = [r["metrics"]["overall"].get("keyword_recall", 0) for r in successful_models]
    throughputs = [r["metrics"]["overall"].get("throughput", 0) for r in successful_models]
    vram_usages = [r["metrics"]["overall"].get("vram_max_allocated_gb", 0) for r in successful_models]
    
    print(f"Exact Match - Avg: {sum(exact_matches)/len(exact_matches):.4f}, Min: {min(exact_matches):.4f}, Max: {max(exact_matches):.4f}")
    print(f"Keyword Recall - Avg: {sum(keyword_recalls)/len(keyword_recalls):.4f}, Min: {min(keyword_recalls):.4f}, Max: {max(keyword_recalls):.4f}")
    print(f"Throughput - Avg: {sum(throughputs)/len(throughputs):.2f} img/s, Min: {min(throughputs):.2f}, Max: {max(throughputs):.2f}")
    print(f"VRAM Usage - Avg: {sum(vram_usages)/len(vram_usages):.2f} GB, Min: {min(vram_usages):.2f}, Max: {max(vram_usages):.2f}")

print("\n" + "="*60)